<a href="https://colab.research.google.com/github/abhsrivastava/hugging_face_transformers/blob/main/Gradio_ChatInterface_and_Speech_to_Text_with_Streaming.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# In this example, we will use the Gradio Chat Interface and use the Whisper Models to build a speech to text chatbot

In [1]:
%pip install -q gradio huggingface_hub "pandas<3.0.0"

print(f'✅ Installed Dependencies Successfully!')

✅ Installed Dependencies Successfully!


In [2]:
# Imports
import gradio as gr
from huggingface_hub import InferenceClient
import os
import warnings
warnings.filterwarnings('ignore')
print(f'Gradio Version {gr.__version__}')
print(f'✅ Imports Successful')

Gradio Version 6.20.0
✅ Imports Successful


In [3]:
# Initialize the InferenceClient with fallback models
# Some models may not be available on the free serverless API at all times
MODELS_TO_TRY = [
    "meta-llama/Llama-3.1-8B-Instruct",
    "Qwen/Qwen2.5-7B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "meta-llama/Llama-3.3-70B-Instruct",
]

client = None
ACTIVE_MODEL = None

for model_id in MODELS_TO_TRY:
  try:
    test_client = InferenceClient(model=model_id)
    test_client.chat_completion(
        [{"role": "user", "content": "hi"}], max_tokens=5
    )
    client = test_client
    print(f'Successfully connected to {model_id}')
    break
  except Exception as ex:
    print(f"⚠️  {model_id} unavailable: {type(e).__name__}")

if client is None:
  print("\n❌ No models available. Check your HF token or try again later.")
else:
  print("💡 This model runs on HF servers — no local GPU needed!")


Successfully connected to meta-llama/Llama-3.1-8B-Instruct
💡 This model runs on HF servers — no local GPU needed!


In [10]:
def extract_text(content):
    """Convert Gradio 6 structured content into plain text."""
    if isinstance(content, str):
        return content

    if isinstance(content, list):
        text_parts = []

        for block in content:
            if isinstance(block, dict):
                if block.get("type") == "text":
                    text_parts.append(block.get("text", ""))
            else:
                text_parts.append(str(block))

        return "\n".join(text_parts)

    return str(content)


def respond(message, history):
    if client is None:
        yield "No Hugging Face inference model is currently available."
        return

    messages = [
        {
            "role": "system",
            "content": (
                "You are NovaPay's customer support assistant. "
                "Be helpful, concise, and professional."
            ),
        }
    ]

    # Gradio 6 supplies OpenAI-style message dictionaries
    for history_message in history:
        role = history_message.get("role")
        content = extract_text(history_message.get("content", ""))

        if role in {"user", "assistant"} and content:
            messages.append(
                {
                    "role": role,
                    "content": content,
                }
            )

    # Add the current message as a dictionary—not a nested list
    messages.append(
        {
            "role": "user",
            "content": message,
        }
    )

    partial = ""

    try:
        for chunk in client.chat_completion(
            messages=messages,
            max_tokens=256,
            stream=True,
        ):
            token = chunk.choices[0].delta.content or ""
            partial += token
            yield partial

    except Exception as ex:
        yield f"Model request failed: {type(ex).__name__}: {ex}"

In [11]:
demo = gr.ChatInterface(
    fn=respond,
    title='NovaPay Support ChatBot v2',
    description="Now with streaming! Tokens appear progressively. Same model, dramatically better UX."
)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2d1c4047418b1dbe3a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [7]:
demo.close()

Closing server running on port: 7860
